In [10]:
from pyClarion import (Agent, Input, Choice, Pool, Event, ChunkStore, NumDict, 
    Priority)
from pyClarion.components.layers import Mapping
from pyClarion.events import Update, ForwardUpdate
from pyClarion.components.rules import RuleStore
from pyClarion.components.io import Controller
from pyClarion.knowledge import (Root, Chunk, ChunkFamily, RuleFamily, DataFamily, 
    AtomFamily, BusFamily, Rule, Atoms, Atom, Buses, Bus)
from pyClarion.structures.buffers import Stack, StackOps
from datetime import timedelta


class MainBuses(Buses):
    acs: Bus       # main bus (for rule-based processing)
    stage: Bus     # memory bus (for stack staging)
    wm: Bus        # wm bus (for stack retrievals)

        
class ModelLayout(BusFamily):
    main: MainBuses

            
class ModelKeyspace[D: DataFamily](Root):
    b:        ModelLayout        # A namespace for the model buses (acs stack_m stack_b)
    c:        ChunkFamily        # A namespace for non-stack chunks (e.g. rule antecedents, consequents)
    p:        AtomFamily         # A namespace for internal parameter symbols (e.g. n, v, nil)
    r:        RuleFamily         # A namespace for rules (model.prs)
    d:        D                  # A (generic) namespace for the model data

    def __init__(self, data_type: type[D]) -> None:
        super().__init__()
        self.d = self["d"] = data_type()

class Model[D: DataFamily](Agent):
    ks: ModelKeyspace[D]
    input_name: str
    
    def __init__(
        self,  
        name: str, 
        data_type: type[D],
        input: str = "input",
        f: float = 1
    ) -> None:
                
        ks = ModelKeyspace(data_type)
        input_sort = ks.d[input]
        assert isinstance(input_sort, Atoms)
        super().__init__(name, ks)
        self.ks = ks
        self.input_name = input
        
        with self:

            # We start by defining knowledge stores for instantiating a simple 
            # system of action rules in pyClarion. 

            # We define two chunk stores for holding, respectively, the 
            # left-hand sides (i.e., antecedents) and right-hand sides (i.e., 
            # consequents) of our rules.
            
            self.lhs = ChunkStore(f"{name}.lhs", ks.c, (ks.b.main, ks.d))
            self.rhs = ChunkStore(f"{name}.rhs", ks.c, (ks.b.main, ks.d))
            
            # We also define a rule store, which provides explicit symbolic 
            # representations of individual rules and defines their linkages 
            # to antecedent and consequent chunks.
            
            self.prs = RuleStore(f"{name}.prs", ks.r, self.lhs.c, self.rhs.c)
            
            self.parse_stack = Stack(
                "parse_stack",
                p=ks.p,
                c=ks.c,               # Namespace of stack chunks
                a=ks.b.main.acs,      # Namespace of action bus — controller reads this
                m=ks.b.main.stage,    # Namespace of staging bus 
                b=ks.b.main.wm,       # Namespace of output bus 
                v=ks.d,               # Namespace of stack vocabulary (data)
                s=ks.d.stack_ops      # Namespace of stack operations
            )

            # We then define a minimal set of components for processing rules. 
            # These components power the model's procssing cycle. The details 
            # of this cycle will become more understandable as the semester 
            # progresses, so for now, focus on developing a high-level 
            # understanding and do not worry too much about the details.
            # 
            # First, we compute the match between the model's input state and 
            # rule antecedents. The rules then compete to be fired, with the 
            # best matching rule having the highest probability of firing. In AI 
            # lingo, this process is called *conflict resolution*. There are 
            # different ways to do conflict resolution; here we stochastically 
            # (i.e., randomly) choose a rule based on how well the model's 
            # input state, as a whole, matches its antecedent and we say that 
            # the chosen rule is *fired*. (If you are wondering how we actually 
            # do this, it is very simple and, yet, we will spend two-to-three 
            # lectures talking about it.) The selected rule then activates its 
            # consequent, which, in turn, triggers the corresponding state 
            # transition.
            #
            # The code below only generates the desired components. They are 
            # not yet connected to each other. That is what we will do next.

            self.input = Input(f"{name}.input", (ks.b.main.acs, input_sort))
            
            self.state = Choice(f"{name}.state", 
                p=ks.p, s=ks.d, d=(ks.b.main.acs, ks.d), sd=1e-3, f=f)
            
            self.inhib = Mapping(f"{name}.inhib", 
                i=(ks.b.main.acs, ks.d), o=(ks.b.main.acs, input_sort),
                func=NumDict.neg)
            
            self.pool_in = Pool(f"{name}.pool_in", 
                p=ks.p, d=(ks.b.main, ks.d), agg=NumDict.sum)
            
            self.lhs_bu = self.lhs.bottom_up(f"{name}.lhs_bu")
            self.lhs_layer = self.prs.lhs_layer(f"{name}.lhs_layer")
            
            self.rule_selector = Choice(f"{name}.rule_selector", 
                p=ks.p, s=ks.d, d=self.prs.r, sd=1e-3)
            
            self.rhs_layer = self.prs.rhs_layer(f"{name}.rhs_layer")
            self.rhs_td = self.rhs.top_down(f"{name}.rhs_td")
            
            self.pool_out = Pool(f"{name}.pool_out", 
                p=ks.p, d=(ks.b.main, ks.d), agg=NumDict.sum)
            
            self.stack_selector = Choice(f"{name}.stack_selector", 
                p=ks.p, s=ks.d, d=(ks.b.main.acs, ks.pool_out), sd=1e-3, f=f)

        # Below, we connect up all the pieces (i.e., assemble the model). As 
        # you can see, this model is relatively big, but not too 
        # complicated. The input and state togther drive rule selection 
        # which, in turn influences the state.
        #
        # The '>>' syntax helps with model assembly. Typically, you pass it 
        # input components on the left and a unique output component on the 
        # right. It returns the output component.
            
        self.inhib = self.state >> self.inhib 
        
        self.rhs_td =(
            (self.input, self.state, self.inhib) 
            >> self.pool_in 
            >> self.lhs_bu 
            >> self.lhs_layer 
            >> self.rule_selector 
            >> self.rhs_layer 
            >> self.rhs_td)
        
        self.state = (
            (self.rhs_td, self.state, self.inhib) 
            >> self.pool_out 
            >> self.state)
        
        self.parse_stack = self.pool_out >> self.parse_stack
        
        stack.parse_stack.buffer.reader = self.pool_out >> stack.parse_stack.buffer.reader
                
        # Choice process from pool_out to s_ops
        # Action bus is (b.main.acs, d.s_ops)
                
        # The following segment is quite important for this tutorial. One 
        # detail about this configuration that I didn't explain just yet is 
        # that we are configuring the model to have some degree of state 
        # inertia. What I mean by this is that we want the model to hold on to 
        # its state unless a rule explicitly recommends changing it. From a 
        # technical point of view, this is achieved by routing information 
        # about the previous state to the selector for the next state. The 
        # weight settings below ensure that this information will be present, 
        # but never dominate recommendations coming from the rules.

        with self.pool_out.params[0].mutable():
            self.pool_out.params[0][~self.pool_out.p[self.state.name]] = 0.5
            self.pool_out.params[0][~self.pool_out.p[self.inhib.name]] = 0.5
            
        
        # This following bit is an essential piece of book-keeping. 
        # 
        # Our model may automatically generate some chunks and rules when we 
        # populate it with knowledge. For this, it will need to know how to 
        # automatically name these new objects. So, here, we inform the 
        # relevant symbolic structures on how to generate names. At some point, 
        # I need to streamline this process but, for now, this is how you do it.

        def namer():
            from itertools import count
            counter = count()
            for i in counter:
                yield str(f"_{i}")
        
        self.lhs.c._namer_ = namer()
        self.rhs.c._namer_ = namer()
        self.prs.r._namer_ = namer()
        self.ks.c_stack._namer_ = namer()
        

    # Many of the components in this model will automatically trigger forward 
    # updates (i.e., updates that propagate information from their inputs to 
    # their outputs). However, some of them do not do this automatically since 
    # the timing of their updates may vary according to their role in a model. 
    # In our resolve method, we specify when these components should propagate 
    # information.

    def resolve(self, event: Event) -> None:
        if event.source == self.input.send:
            self.system.schedule(self.pool_in.forward())
        if event.source == self.lhs_layer.forward:
            self.system.schedule(self.rule_selector.trigger())
        if event.source == self.rhs_td.forward:
            self.system.schedule(self.pool_out.forward())
        if event.source == self.pool_out.forward:
            self.system.schedule(self.state.trigger())


    # Next, we define a method to help us with simulation initialization. This 
    # init method is what you might call an 'event method' because it generates 
    # an event that you can put into your simulation.
    # 
    # This method implementation is a peek into more advanced pyClarion 
    # programming. Here is what it does: If given a specific set of states, 
    # initialize the model with those states, otherwise randomly select initial 
    # state symbols using random.shuffle(). 
    #
    # The parameters, dt and priority, are standard in pyClarion event methods 
    # (though, strictly speaking, not required). They allow users to exert 
    # fine-grained control on the event scheduling process. I've included them 
    # here for reference; we will not actually use them in our simulations for 
    # this tutorial.

    def init(self, 
        *states: Atom, 
        dt: timedelta = timedelta(), 
        priority: Priority = Priority.PROPAGATION
    ) -> Event:
        bus = self.ks.b.main.acs
        if not states:
            import random
            _states = []
            for dim_name in self.ks.d:
                if dim_name != self.input_name:
                    dim = self.ks.d[dim_name]
                    options = list(dim)
                    random.shuffle(options)
                    _states.append(dim[options[0]])
            states = tuple(_states)
        data = {~bus * ~state: 1.0 for state in states}
        ud: list[Update] = [ForwardUpdate(self.state.main, data)]
        return Event(self.init, ud, dt, priority)

In [11]:
class POS(Atoms):
    nil: Atom
    det: Atom
    n: Atom
    v_main: Atom
    v_part: Atom
    p: Atom
    end: Atom

class Expect(Atoms):
    nil: Atom
    det: Atom
    n: Atom
    v: Atom
    p: Atom

class Phrase(Atoms):
    nil: Atom
    np: Atom
    vp: Atom
    vp_part: Atom
    pp: Atom

class StackCmd(Atoms):
    nil: Atom
    push: Atom
    pop: Atom
    flush: Atom

class ParseState(Atoms):
    start: Atom
    run: Atom
    error: Atom
    stop: Atom

class ParserData(DataFamily):
    input:  POS
    expect: Expect
    phrase: Phrase
    cmd:    StackCmd
    state:  ParseState
    stack_ops: StackOps


def init_phrase_structure_rules(ks: ModelKeyspace[ParserData]) -> list[Rule]:
    b = ks.b
    d = ks.d
    return [
        
        # Start: push S, expect Det
        "start_parse" ^
        + b.main.acs ** d.state.start
        >>
        + b.main.acs ** d.expect.det
        + b.main.acs ** d.phrase.np
        + b.main.acs ** d.cmd.push       # push NP prediction
        + b.main.acs ** d.state.run,

        # Det confirmed
        "confirm_det" ^
        + b.main.acs ** d.input.det
        + b.main.acs ** d.expect.det
        + b.main.acs ** d.state.run
        >>
        + b.main.acs ** d.expect.n
        + b.main.acs ** d.cmd.nil,

        # N confirmed: default to expecting main verb (minimal attachment)
        "confirm_n" ^
        + b.main.acs ** d.input.n
        + b.main.acs ** d.expect.n
        + b.main.acs ** d.state.run
        >>
        + b.main.acs ** d.expect.v
        + b.main.acs ** d.cmd.nil,

        # V_main when expected: push VP, consume
        "confirm_v_main" ^
        + b.main.acs ** d.input.v_main
        + b.main.acs ** d.expect.v
        + b.main.acs ** d.state.run
        >>
        + b.main.acs ** d.phrase.vp
        + b.main.acs ** d.expect.p
        + b.main.acs ** d.cmd.push,

        # V_part when v expected: garden path → flush
        "garden_path" ^
        + b.main.acs ** d.input.v_part
        + b.main.acs ** d.expect.v
        + b.main.acs ** d.state.run
        >>
        + b.main.acs ** d.cmd.flush
        + b.main.acs ** d.state.error,

        # V_part on reanalysis pass (expect overridden by loop after flush)
        "confirm_v_part" ^
        + b.main.acs ** d.input.v_part
        + b.main.acs ** d.expect.v
        + b.main.acs ** d.phrase.vp_part
        + b.main.acs ** d.state.run
        >>
        + b.main.acs ** d.expect.p
        + b.main.acs ** d.cmd.nil,

        # P confirmed: push PP, recurse into inner NP
        "confirm_p" ^
        + b.main.acs ** d.input.p
        + b.main.acs ** d.expect.p
        + b.main.acs ** d.state.run
        >>
        + b.main.acs ** d.phrase.pp
        + b.main.acs ** d.expect.det
        + b.main.acs ** d.cmd.push,

        # Inner NP complete: pop PP, pop VP/VP_part, back to open NP
        "finish_pp" ^
        + b.main.acs ** d.input.v_main
        + b.main.acs ** d.expect.v
        + b.main.acs ** d.phrase.np
        + b.main.acs ** d.state.run
        >>
        + b.main.acs ** d.phrase.vp
        + b.main.acs ** d.expect.nil
        + b.main.acs ** d.cmd.pop
        + b.main.acs ** d.state.stop,

        # End of input
        "finish_parse" ^
        + b.main.acs ** d.input.end
        + b.main.acs ** d.state.run
        >>
        + b.main.acs ** d.cmd.nil
        + b.main.acs ** d.state.stop,

        # Inertial rule
        "inertial" ^
        + b.main.acs ** d.state("X")
        >>
        + b.main.acs ** d.state("X"),

    ]

# def init_stack_chunks(ks: ModelKeyspace[ParserData]) -> list[Chunk]:
#     b = ks.b
#     d = ks.d
#     return [
#         Chunk({(b.main.stack_b, d.phrase.np):      1.0}),
#         Chunk({(b.main.stack_b, d.phrase.vp):      1.0}),
#         Chunk({(b.main.stack_b, d.phrase.vp_part): 1.0}),
#         Chunk({(b.main.stack_b, d.phrase.pp):      1.0}),
#     ]

In [12]:
# Initialize model and build the simulation loop.
model = Model("model", data_type=ParserData, f=.1)

# Encode all rules into the model's memory
parser_rules = init_phrase_structure_rules(model.ks)
model.system.schedule(model.prs.encode(*parser_rules))
model.system.run_all()

# Encode all possible stack contents
stack_chunks = init_stack_chunks(model.ks)
model.system.schedule(model.parse_stack.chunks.encode(*stack_chunks))
model.system.run_all()

# Dimension keys for polling

# Define which slice of the ACS buffer you want to read from.
# e.g. ~main_acs * ~model.ks.d.state is a dimension key that says
# “Give me the activation of the state dimension on the ACS bus.”

main_acs   = model.ks.b.main.acs
state_dim  = ~main_acs * ~model.ks.d.state
cmd_dim    = ~main_acs * ~model.ks.d.cmd
expect_dim = ~main_acs * ~model.ks.d.expect
phrase_dim = ~main_acs * ~model.ks.d.phrase

# Convert keys back into readable atoms.
# e.g (~b.main.acs * ~d.state.start) → d.state.start
state_coder  = {~main_acs * ~model.ks.d.state[n]:  n for n in model.ks.d.state}
cmd_coder    = {~main_acs * ~model.ks.d.cmd[n]:    n for n in model.ks.d.cmd}
expect_coder = {~main_acs * ~model.ks.d.expect[n]: n for n in model.ks.d.expect}
phrase_coder = {~main_acs * ~model.ks.d.phrase[n]: n for n in model.ks.d.phrase}

# The ACS key corresponding to the error state (our trigger for reanalysis)
error_key = ~main_acs * ~model.ks.d.state.error

# Stimulus handles
#
det    = + model.ks.b.main.acs ** model.ks.d.input.det
n      = + model.ks.b.main.acs ** model.ks.d.input.n
v_main = + model.ks.b.main.acs ** model.ks.d.input.v_main
v_part = + model.ks.b.main.acs ** model.ks.d.input.v_part
p      = + model.ks.b.main.acs ** model.ks.d.input.p
end    = + model.ks.b.main.acs ** model.ks.d.input.end

stimulus_coder = {
    det:    "det",
    n:      "n",
    v_main: "v_main",
    v_part: "v_part",
    p:      "p",
    end:    "end",
}

sentence_pass1 = [det, n, v_main, p, det, n, v_main, end]
sentence_pass2 = [det, n, v_part, p, det, n, v_main, end]

results = {
    "pass":   [],
    "step":   [],
    "word":   [],
    "input":  [],
    "expect": [],
    "phrase": [],
    "state":  [],
    "cmd":    [],
    "stack":  [],
}

def record(pass_num, step, word, stimulus, poll, stack_contents):
    results["pass"].append(pass_num)
    results["step"].append(step)
    results["word"].append(word)
    results["input"].append(stimulus_coder.get(stimulus, ""))
    results["expect"].append(expect_coder[poll[expect_dim]])
    results["phrase"].append(phrase_coder[poll[phrase_dim]])
    results["state"].append(state_coder[poll[state_dim]])
    results["cmd"].append(cmd_coder[poll[cmd_dim]])
    results["stack"].append(" > ".join(stack_contents) if stack_contents else "[ ]")


words_pass1 = ["the", "horse", "raced", "past", "the", "barn", "fell", "END"]
words_pass2 = ["the", "horse", "raced", "past", "the", "barn", "fell", "END"]

def parse(stimuli, words, pass_num):
    stack_contents = []

    model.system.schedule(model.init(
        model.ks.d.state.start,
        model.ks.d.expect.nil,
        model.ks.d.phrase.nil,
        model.ks.d.cmd.nil,
    ))
    model.system.run_all()

    poll = model.state.poll()
    record(pass_num, 0, "—", None, poll, stack_contents)

    for step, (s, word) in enumerate(zip(stimuli, words), start=1):
        model.system.schedule(model.input.send(s))
        model.system.run_all()

        poll = model.state.poll()
        cmd  = poll[cmd_dim]

        if cmd == ~main_acs * ~model.ks.d.cmd.push:
            phrase = phrase_coder[poll[phrase_dim]]
            stack_contents = [phrase] + stack_contents
            # Write current phrase onto stack_m so the buffer has something to stage
            phrase_atom = model.ks.d.phrase[phrase]
            model.system.schedule(model.stack_input.send(+ model.ks.b.main.stack_m ** phrase_atom))
            model.system.run_all()
            model.system.schedule(model.parse_stack.start_push())
            model.system.run_all()
        elif cmd == ~main_acs * ~model.ks.d.cmd.pop:
            if stack_contents:
                stack_contents = stack_contents[1:]
            model.system.schedule(model.parse_stack.start_pop())
            model.system.run_all()
        elif cmd == ~main_acs * ~model.ks.d.cmd.flush:
            stack_contents = []
            model.system.schedule(model.parse_stack.start_flush())
            model.system.run_all()

        record(pass_num, step, word, s, poll, stack_contents)

        if poll[state_dim] == error_key:
            return False

    return True


print("Pass 1:")
success = parse(sentence_pass1, words_pass1, pass_num=1)
if not success:
    print("Garden path detected — reanalysing\n")
    print("Pass 2:")
    parse(sentence_pass2, words_pass2, pass_num=2)


# Display results
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
data = pd.DataFrame(results)
data

AttributeError: 'ModelKeyspace' object has no attribute 'pool_out'